In [1]:
import pandas as pd

# Load the ratings/recommendation dataset
ratings_df = pd.read_csv("../data/movies_dataset.csv")

# Load the movie metadata dataset
metadata_df = pd.read_csv("../data/movies_updated-selected-columns.csv")

print("Ratings dataset:", ratings_df.shape)
print("Metadata dataset:", metadata_df.shape)

Ratings dataset: (25000095, 5)
Metadata dataset: (4000, 10)


In [2]:
print("Ratings columns:")
print(ratings_df.columns.tolist())

print("\nMetadata columns:")
print(metadata_df.columns.tolist())

Ratings columns:
['Unnamed: 0', 'User_Id', 'Movie_Name', 'Rating', 'Genre']

Metadata columns:
['name', 'rating', 'genre', 'year', 'released', 'score', 'votes', 'director', 'writer', 'star']


In [3]:
rating_movies = ratings_df[
    ["Movie_Name"]
].drop_duplicates()

print("Unique movies in ratings dataset:", len(rating_movies))

Unique movies in ratings dataset: 58958


In [4]:
rating_movies["year"] = (
    rating_movies["Movie_Name"]
    .str.extract(r"\((\d{4})\)$")
)

In [5]:
rating_movies["clean_title"] = (
    rating_movies["Movie_Name"]
    .str.replace(r"\s*\(\d{4}\)$", "", regex=True)
    .str.strip()
)

In [6]:
metadata_df["clean_title"] = (
    metadata_df["name"]
    .astype(str)
    .str.strip()
)

In [7]:
metadata_df["year"] = pd.to_numeric(
    metadata_df["year"],
    errors="coerce"
)

In [8]:
print("Ratings titles:")
print(rating_movies[["clean_title", "year"]].head(10))

print("\nMetadata titles:")
print(metadata_df[["clean_title", "year"]].head(10))

Ratings titles:
                                 clean_title  year
0                               Pulp Fiction  1994
1  Three Colors: Red (Trois couleurs: Rouge)  1994
2  Three Colors: Blue (Trois couleurs: Bleu)  1993
3                                Underground  1995
4                        Singin' in the Rain  1952
5                              Dirty Dancing  1987
6                               Delicatessen  1991
7                                        Ran  1985
8   Seventh Seal, The (Sjunde inseglet, Det)  1957
9              Bridge on the River Kwai, The  1957

Metadata titles:
                                      clean_title  year
0                                     The Shining  1980
1                                 The Blue Lagoon  1980
2  Star Wars: Episode V - The Empire Strikes Back  1980
3                                       Airplane!  1980
4                                      Caddyshack  1980
5                                 Friday the 13th  1980
6            

In [9]:
rating_movies["year"] = pd.to_numeric(
    rating_movies["year"],
    errors="coerce"
)

metadata_df["year"] = pd.to_numeric(
    metadata_df["year"],
    errors="coerce"
)

In [10]:
print("Ratings year type:", rating_movies["year"].dtype)
print("Metadata year type:", metadata_df["year"].dtype)

Ratings year type: float64
Metadata year type: int64


In [11]:
rating_movies["year"] = rating_movies["year"].astype("Int64")
metadata_df["year"] = metadata_df["year"].astype("Int64")

In [12]:
print("Ratings year type:", rating_movies["year"].dtype)
print("Metadata year type:", metadata_df["year"].dtype)

Ratings year type: Int64
Metadata year type: Int64


In [13]:
exact_matches = rating_movies.merge(
    metadata_df[["clean_title", "year"]],
    on=["clean_title", "year"],
    how="inner"
)

print("Exact matches:", len(exact_matches))

Exact matches: 2759


In [14]:
match_percentage = (
    len(exact_matches) / len(rating_movies)
) * 100

print(f"Exact match percentage: {match_percentage:.2f}%")

Exact match percentage: 4.68%


In [15]:
import re

def normalize_title(title):
    title = str(title).lower().strip()

    # Move "the" from the end to the beginning
    title = re.sub(r",\s*the$", "", title)

    # Remove punctuation
    title = re.sub(r"[^a-z0-9\s]", "", title)

    # Normalize spaces
    title = re.sub(r"\s+", " ", title).strip()

    return title

In [16]:
rating_movies["normalized_title"] = (
    rating_movies["clean_title"]
    .apply(normalize_title)
)

In [17]:
metadata_df["normalized_title"] = (
    metadata_df["clean_title"]
    .apply(normalize_title)
)

In [18]:
normalized_matches = rating_movies.merge(
    metadata_df[
        ["normalized_title", "year"]
    ],
    on=["normalized_title", "year"],
    how="inner"
)

print("Normalized matches:", len(normalized_matches))

Normalized matches: 2835


In [19]:
normalized_percentage = (
    len(normalized_matches) / len(rating_movies)
) * 100

print(
    f"Normalized match percentage: "
    f"{normalized_percentage:.2f}%"
)

Normalized match percentage: 4.81%


In [20]:
model_movies = pd.read_csv(
    "../data/model_movies.csv"
)

print("Model movies:", len(model_movies))

Model movies: 3794


In [21]:
model_movies["year"] = (
    model_movies["Movie_Name"]
    .str.extract(r"\((\d{4})\)$")
)

model_movies["clean_title"] = (
    model_movies["Movie_Name"]
    .str.replace(
        r"\s*\(\d{4}\)$",
        "",
        regex=True
    )
    .str.strip()
)

model_movies["year"] = pd.to_numeric(
    model_movies["year"],
    errors="coerce"
).astype("Int64")

model_movies["normalized_title"] = (
    model_movies["clean_title"]
    .apply(normalize_title)
)

In [22]:
model_matches = model_movies.merge(
    metadata_df[
        ["normalized_title", "year"]
    ],
    on=["normalized_title", "year"],
    how="inner"
)

print("Model movies:", len(model_movies))
print("Metadata matches:", len(model_matches))

percentage = (
    len(model_matches) /
    len(model_movies)
) * 100

print(f"Model metadata coverage: {percentage:.2f}%")

Model movies: 3794
Metadata matches: 1283
Model metadata coverage: 33.82%


In [23]:
matched_titles = set(model_matches["normalized_title"])

unmatched_movies = model_movies[
    ~model_movies["normalized_title"].isin(matched_titles)
].copy()

print("Unmatched model movies:", len(unmatched_movies))

Unmatched model movies: 2487


In [24]:
unmatched_movies[
    ["Movie_Name", "year"]
].head(30)

,Movie_Name,year
6,Star Wars: Episode IV - A New Hope (1977),1977
9,"Shawshank Redemption, The (1994)",1994
13,"Lion King, The (1994)",1994
15,"Fugitive, The (1993)",1993
25,"Rock, The (1996)",1996
26,"Godfather, The (1972)",1972
27,My Fair Lady (1964),1964
28,It's a Wonderful Life (1946),1946
29,"Sound of Music, The (1965)",1965
30,Monty Python's Life of Brian (1979),1979


In [25]:
def normalize_title(title):
    title = str(title).lower().strip()

    # Move ", the" from the end to the beginning
    if re.search(r",\s*the$", title):
        title = re.sub(r",\s*the$", "", title)
        title = "the " + title

    # Remove punctuation
    title = re.sub(r"[^a-z0-9\s]", "", title)

    # Normalize spaces
    title = re.sub(r"\s+", " ", title).strip()

    return title

In [26]:
model_movies["normalized_title"] = (
    model_movies["clean_title"]
    .apply(normalize_title)
)

metadata_df["normalized_title"] = (
    metadata_df["clean_title"]
    .apply(normalize_title)
)

In [27]:
model_matches = model_movies.merge(
    metadata_df[
        ["normalized_title", "year"]
    ],
    on=["normalized_title", "year"],
    how="inner"
)

print("Model movies:", len(model_movies))
print("Metadata matches:", len(model_matches))

percentage = (
    len(model_matches) /
    len(model_movies)
) * 100

print(f"Model metadata coverage: {percentage:.2f}%")

Model movies: 3794
Metadata matches: 1572
Model metadata coverage: 41.43%


In [28]:
credits_df = pd.read_csv(
    "../data/credits.csv"
)

print("Rows:", len(credits_df))
print("Columns:", credits_df.columns.tolist())
print("Unique movie IDs:", credits_df["movie_id"].nunique())
print("Duplicate titles:", credits_df["title"].duplicated().sum())

Rows: 4803
Columns: ['movie_id', 'title', 'cast', 'crew']
Unique movie IDs: 4803
Duplicate titles: 3


In [29]:
credits_df[["movie_id", "title"]].head(20)

,movie_id,title
0,19995,Avatar
1,285,Pirates of the Caribbean: At World's End
2,206647,Spectre
3,49026,The Dark Knight Rises
4,49529,John Carter
5,559,Spider-Man 3
6,38757,Tangled
7,99861,Avengers: Age of Ultron
8,767,Harry Potter and the Half-Blood Prince
9,209112,Batman v Superman: Dawn of Justice


In [30]:
credits_df["movie_id"].head(20).tolist()

[19995,
 285,
 206647,
 49026,
 49529,
 559,
 38757,
 99861,
 767,
 209112,
 1452,
 10764,
 58,
 57201,
 49521,
 2454,
 24428,
 1865,
 41154,
 122917]

In [31]:
credits_df["movie_id"].describe()

count      4803.000000
mean      57165.484281
std       88694.614033
min           5.000000
25%        9014.500000
50%       14629.000000
75%       58610.500000
max      459488.000000
Name: movie_id, dtype: float64